<a href="https://colab.research.google.com/github/nyp-sit/dit-it3103/blob/main/week4/3.fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT3103 Practical 04c: Fine-tuning

Another widely used transfer learning technique is _fine-tuning_.
Fine-tuning involves unfreezing a few of the top layers
of a frozen model base used for feature extraction, and jointly training both the newly added part of the model (in our case, the
fully-connected classifier) and these unfrozen top layers. This is called "fine-tuning" because it slightly adjusts the more abstract
representations of the model being reused, in order to make them more relevant for the problem at hand.

![fine-tuning VGG16](https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/resources/vgg16_fine_tuning.png)

In [ ]:
import os
import keras
import keras.applications

It was necessary to freeze the convolution base of VGG16 in order to be able to train a randomly initialized
classifier on top. For the same reason, it is only possible to fine-tune the top layers of the convolutional base once the classifier on
top has already been trained. If the classified wasn't already trained, then the error signal propagating through the network during
training would be too large, and the representations previously learned by the layers being fine-tuned would be destroyed. Thus the steps
for fine-tuning a network are as follow:

1. Add your custom network on top of an already trained base network.
2. Freeze the base network.
3. Train the part you added.
4. Unfreeze some layers in the base network.
5. Jointly train both these layers and the part you added.


## Load the pre-trained VGG16 model excluding the classification header

In [ ]:
img_height, img_width = 128, 128

# Load the pre-trained model
base_model = keras.applications.VGG16(input_shape=(img_height, img_width) + (3,),
                                         include_top=False,
                                         weights='imagenet')

preprocess_input_fn = keras.applications.vgg16.preprocess_input

# freeze the base layer
base_model.trainable = False

# Add input layer
inputs = keras.layers.Input(shape=(img_height, img_width, 3))
# Add preprocessing layer
x = preprocess_input_fn(inputs)
# Add the base, set training to false to freeze the convolutional base
x = base_model(x)
# Add our classification head
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(rate=0.5)(x)
x = keras.layers.Dense(units=512, activation="relu")(x)
x = keras.layers.Dropout(rate=0.5)(x)
outputs = keras.layers.Dense(units=1, activation="sigmoid")(x)

model = keras.models.Model(inputs=[inputs], outputs=[outputs])

base_learning_rate = 0.001

model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=base_learning_rate),
                  metrics=["accuracy"])


Let's confirm all the layers of convolutional base are frozen.

In [ ]:
for layer in base_model.layers:
    print(layer.name, layer.trainable)

Let's print out the model summary and see how many trainable weights. We can see that we only 263,169 trainable weights (parameters), coming from the classification head that put on top of the convolutional base. (For comparison, a VGG16 has total of 14,714,688 weights).

In [ ]:
model.summary()

## Download Datasets

We will setup our training and validation dataset as we did in earlier exercise.

In [ ]:
dataset_URL = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/cats_and_dogs_subset.tar.gz'
path_to_zip = keras.utils.get_file('cats_and_dogs_subset.tar.gz', origin=dataset_URL, extract=True, cache_dir='.')
dataset_dir = os.path.join(os.path.dirname(path_to_zip), "cats_and_dogs_subset_extracted/cats_and_dogs_subset")

In [ ]:
batch_size = 32
image_size = (img_height, img_width)

train_ds = keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
    label_mode='binary'
)
val_ds = keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
    label_mode='binary'
)

## Only train the classification head

We will go ahead and train our classification head.

In [ ]:
# create model checkpoint callback to save the best model checkpoint
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath="best_checkpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

model.fit(train_ds, validation_data=val_ds,
          epochs=30, callbacks=[model_checkpoint_callback])

In [ ]:
model.load_weights('best_checkpoint.weights.h5')
eval_result = model.evaluate(val_ds)
print("[test loss, test accuracy]:", eval_result)

## Fine-tuning last 3 convolutional layers and classification head together

We will go ahead and train our classification head.

Now we have our classification layers trained, let's start to unfreeze some top layers of the convolutional base to fine tune the weights.
We will fine-tune the last 3 convolutional layers, which means that all layers up until `block4_pool` should be frozen, and the layers
`block5_conv1`, `block5_conv2` and `block5_conv3` should be trainable.

Why not fine-tune more layers? Why not fine-tune the entire convolutional base? We could. However, we need to consider that:

* Earlier layers in the convolutional base encode more generic, reusable features, while layers higher up encode more specialized features. It is
more useful to fine-tune the more specialized features, as these are the ones that need to be repurposed on our new problem. There would
be fast-decreasing returns in fine-tuning lower layers.
* The more parameters we are training, the more we are at risk of overfitting. The convolutional base has 15M parameters, so it would be
risky to attempt to train it on our small dataset.

Thus, in our situation, it is a good strategy to only fine-tune the top 2 to 3 layers in the convolutional base.

Let's set this up, we will unfreeze our `base_model`,
and then freeze individual layers inside of it, except the last 3 layers.

Do a model ``summary()`` and you will see now that the number of trainable weights are now 7,079,424 (around 7 millions), much less than previously, because all the layers are frozen except the last 3 layers.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

Let us examine model summary again. We can see now that we have more trainable weights 7,342,593 compared to previously 263,169.

In [ ]:
model.summary()

As you are training a much larger model and want to readapt the pretrained weights, it is important to use a lower learning rate at this stage as we do not want to make too drastic changes to the weights in the convolutional layers under fine-tuning.

In [ ]:
finetune_learning_rate = base_learning_rate / 10.

model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=finetune_learning_rate),
              metrics=["accuracy"])

model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[model_checkpoint_callback])

In [ ]:
model.load_weights('best_checkpoint.weights.h5')
eval_result = model.evaluate(val_ds)
print("[test loss, test accuracy]:", eval_result)

## Exercise 1: Compare Feature Extraction vs Fine-tuning

Compare the validation accuracy you achieved with:
- Feature extraction only (from Practical 03b)
- Fine-tuning the last 3 conv layers (from above)

Which approach performed better? Why do you think that is the case?

<details>
<summary>Click here for answer</summary>

Fine-tuning generally performs better because it allows the top convolutional layers to adapt their learned features to be more specific to our cats vs dogs task. Feature extraction keeps the convolutional base completely fixed, so it can only use the generic features learned from ImageNet without any task-specific adaptation.

</details>

## Exercise 2: Experiment with Unfreezing More Layers

In the example above, we unfroze the last 4 layers (block5_conv1 to block5_pool).

Now try unfreezing **more layers** -- unfreeze from `block4_conv1` onwards (the last 8 layers). Retrain the model and compare.

- Does unfreezing more layers improve or worsen performance?
- What happens to the training time?

Hint: Reload the best checkpoint first, then set `base_model.trainable = True` and freeze all layers except the last 8.

<details>
<summary>Click here for answer</summary>

```python
# Reload best weights from previous fine-tuning
model.load_weights('best_checkpoint.weights.h5')

# Unfreeze more layers (from block4_conv1 onwards)
base_model.trainable = True
for layer in base_model.layers[:-8]:
    layer.trainable = False

for layer in base_model.layers:
    print(f"{layer.name}: trainable={layer.trainable}")

model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    metrics=['accuracy'])

model.fit(train_ds, validation_data=val_ds, epochs=10,
          callbacks=[model_checkpoint_callback])

model.load_weights('best_checkpoint.weights.h5')
eval_result = model.evaluate(val_ds)
print(f'Validation accuracy: {eval_result[1]:.4f}')
```

With a small dataset (3000 images), unfreezing too many layers may lead to overfitting. You may see similar or slightly worse performance. The training time will also increase since more parameters need to be updated.

</details>

In [ ]:
## TODO: Reload the best checkpoint weights


## TODO: Unfreeze base_model and freeze all layers except the last 8


## TODO: Recompile the model with a lower learning rate (e.g., 0.00001)


## TODO: Train the model



In [ ]:
## TODO: Load best weights and evaluate on validation set



## Exercise 3: Try a Different Pretrained Model

Replace VGG16 with **MobileNetV2** as the base model and repeat the fine-tuning process.

- Load MobileNetV2 with `include_top=False` and `weights='imagenet'`
- Use `keras.applications.mobilenet_v2.preprocess_input` for preprocessing
- Freeze the base, train the classification head, then unfreeze the last few layers and fine-tune
- Compare with VGG16: accuracy and training speed

<details>
<summary>Click here for answer</summary>

```python
base_model_mv2 = keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet')

preprocess_mv2 = keras.applications.mobilenet_v2.preprocess_input
base_model_mv2.trainable = False

inputs = keras.layers.Input(shape=(img_height, img_width, 3))
x = preprocess_mv2(inputs)
x = base_model_mv2(x)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(256, activation='relu')(x)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation='sigmoid')(x)

model_mv2 = keras.models.Model(inputs, outputs)
model_mv2.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy'])

# Train classification head
model_mv2.fit(train_ds, validation_data=val_ds, epochs=20)

# Fine-tune last 20 layers
base_model_mv2.trainable = True
for layer in base_model_mv2.layers[:-20]:
    layer.trainable = False

model_mv2.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy'])

model_mv2.fit(train_ds, validation_data=val_ds, epochs=10)
model_mv2.evaluate(val_ds)
```

MobileNetV2 is much faster to train due to fewer parameters and depthwise separable convolutions. It often achieves comparable or better accuracy than VGG16 on this task.

</details>

In [ ]:
## TODO: Load MobileNetV2 as base model (include_top=False, weights='imagenet')


## TODO: Build the full model with classification head


## TODO: Train classification head (freeze base), then fine-tune last few layers


## TODO: Evaluate and compare with VGG16 results

